# Pill classification system — Kaggle T4x2 training notebook

## What this notebook runs (in order)

| Phase | Script | Description |
|---|---|---|
| T1 | `train_dino_ssl.py` | DINO domain adaptation — **9-channel** fused pairs |
| T2+T3 | `train_tower.py` (×3 towers ×3 phases ×N folds) | LOOCV tower training |
| T4 | `train_prototype.py` | Prototypical network — 50k episodes |
| — | `build_profiles_and_ood.py` | Constraint profiles + OOD index from **ROI crops** |
| — | `generate_calibration_signals.py` | LOOCV inference pass → signals.jsonl |
| — | `build_calibrator.py` | Isotonic calibration on real signals |
| — | `export_runtime_pack.py` | Final deployable bundle |

## Dataset folder structure expected

```
your_dataset/
  DrugName_Dosage_A/              ← one folder per class
    pair_001/
      side_a.png
      side_b.png
    pair_002/
      ...
  DrugName_Dosage_B/
    ...
```

No background_reference.png required. The system uses a pure black reference automatically — correct for black-tray pharmacy setups.

Pair grouping: images are grouped into pairs by subfolder name or by filename prefix before the first underscore. Each pair must have exactly 2 images — one per side.

## Bug fixes applied vs original code
- **Bug 1**: Argument mismatch in launcher fixed; build_profiles uses black reference directly
- **Bug 2**: Constraint profiles built from preprocessed ROI crops (not raw frames)
- **Bug 3**: OOD index built from preprocessed ROI features (not raw frame features)
- **Bug 4**: `generate_calibration_signals.py` generates actual signal records before calibration
- **Issue 5**: DINO trains `DINOAdaptBackbone` (9-channel) — adapted weights load cleanly into all tower stems
- **Issue 6**: `LightROIPreprocessor` runs on training images so train/inference distributions match
- **background_reference removed**: black reference (np.zeros) used everywhere, no file needed


In [ ]:
# Cell 1 — install dependencies
%cd /kaggle/working
!pip install -q timm faiss-cpu scikit-learn scikit-image scipy joblib tqdm

In [ ]:
# Cell 2 — upload your codebase zip to Kaggle as a dataset, then adjust this path
import subprocess, os

# Path to the pill_prod_code_side_complete.zip uploaded as a Kaggle dataset
CODE_ZIP = '/kaggle/input/pill-prod-code/pill_prod_code_side_complete.zip'

subprocess.run(['unzip', '-q', '-o', CODE_ZIP, '-d', '/kaggle/working/pill_system'], check=True)

# Move into the trainer directory
%cd /kaggle/working/pill_system/work_exact/trainer
!pip install -q -r requirements.txt
print('Setup complete')

In [ ]:
# Cell 5 — verify output bundle
import os, json

BUNDLE = f'{WORKDIR}/runtime_bundle'
manifest_path = f'{BUNDLE}/pack_manifest.json'
assert os.path.exists(manifest_path), 'pack_manifest.json missing — training may have failed'

manifest = json.loads(open(manifest_path).read())
print('=== Runtime bundle contents ===')
for name, info in manifest['files'].items():
    size_kb = info['bytes'] / 1024
    print(f'  {name:<40}  {size_kb:>10.1f} KB')

# Verify calibrator is not a placeholder (should have >1 training sample)
import joblib, numpy as np
cal = joblib.load(f'{BUNDLE}/calibrator.joblib')
n_cal = len(cal['train_features'])
print(f'\nCalibrator trained on {n_cal} signal records')
assert n_cal > 1, 'Calibrator is a placeholder — signal generation may have failed'

# Show reliability diagram
reliability_path = f'{WORKDIR}/reliability.json'
if os.path.exists(reliability_path):
    rel = json.loads(open(reliability_path).read())
    print('\n=== Reliability diagram ===')
    print(f'{"Bin":>12}  {"Predicted":>10}  {"Actual":>10}  {"Count":>6}')
    for row in rel:
        print(f"{row['bin_lo']:.2f}–{row['bin_hi']:.2f}  {row['mean_pred']:>10.3f}  {row['empirical_acc']:>10.3f}  {row['count']:>6}")

print('\nBundle verified successfully.')

In [ ]:
# Cell 6 — zip and download the runtime bundle
# This is what you deploy to the Windows server running the FastAPI service.

import shutil, os

OUTPUT_ZIP = f'{WORKDIR}/pill_runtime_bundle.zip'
shutil.make_archive(
    base_name=OUTPUT_ZIP.replace('.zip', ''),
    format='zip',
    root_dir=WORKDIR,
    base_dir='runtime_bundle',
)
size_mb = os.path.getsize(OUTPUT_ZIP) / (1024 * 1024)
print(f'Bundle zip: {OUTPUT_ZIP}  ({size_mb:.1f} MB)')
print('Download this file from the Kaggle output panel.')